
# Phase A — BILL_AMT1 Experimental Audit

## Objective

Verify the existing SHML experimental results and establish a trustworthy baseline before conducting rigorous missingness and recovery experiments.

## Audit Questions

1. Are the dataset splits correct and reproducible?
2. Does the healthy baseline reproduce the recorded metrics?
3. Is preprocessing fitted exclusively on training data?
4. Why do exploratory and formal experiments report different F1-scores?
5. Were model predictions evaluated before recovery?
6. Are safety decisions calculated from actual metrics?
7. Are the experimental thresholds and random seeds consistent?

## Expected Outcome

A verified experimental configuration, documented discrepancies, and a reliable foundation for subsequent BILL_AMT1 research.


In [3]:
from pathlib import Path
import pandas as pd

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    p for p in (cwd, *cwd.parents)
    if (p / "data" / "working").is_dir()
)

DATA_PATH = (
    PROJECT_ROOT / "data" / "working"
    / "default of credit card clients.xls"
)

df = pd.read_excel(DATA_PATH, header=1, engine="xlrd")

print("Dataset shape:", df.shape)
print("Number of missing values:", df.isna().sum().sum())
print("Duplicate IDs:", df["ID"].duplicated().sum())

print("\nBILL_AMT1 summary:")
print(df["BILL_AMT1"].describe())

Dataset shape: (30000, 25)
Number of missing values: 0
Duplicate IDs: 0

BILL_AMT1 summary:
count     30000.000000
mean      51223.330900
std       73635.860576
min     -165580.000000
25%        3558.750000
50%       22381.500000
75%       67091.000000
max      964511.000000
Name: BILL_AMT1, dtype: float64


Three important observations:
1. The mean is substantially higher than the median. This suggests the feature is right-skewed, potentially influenced by large billing amounts.
2. Negative billing amounts exist. They should not automatically be treated as errors; they may represent credits or adjustments.
3. The full-dataset median is 22,381.50. Our previous SHML pipeline used a training-set median of 22,225.50. This is not necessarily a discrepancy: training-only statistics differ from full-dataset statistics, and using the training median helps prevent data leakage.


## Step 2 — Data Splitting Verification

Reproduce the original stratified train/validation/test
split using random seed 42.

Verify that the resulting datasets contain the expected
number of observations and preserve the target distribution.


In [5]:
from sklearn.model_selection import train_test_split

TARGET = "default payment next month"
X = df.drop(columns = ["ID", TARGET])
y = df[TARGET]

# Original training split: 70%
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# Split remaining 30% equally
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training observations:", len(X_train))
print("Validation observations:", len(X_val))
print("Testing observations:", len(X_test))

print("\nTarget class distribution:")
for name, target in [
    ("Train", y_train),
    ("Validation", y_val),
    ("Test", y_test)
]:
    print(name, target.value_counts(normalize=True).round(4).to_dict())

print("\nTraining BILL_AMT1 median:",
      X_train["BILL_AMT1"].median())

assert len(X_train) == 21000
assert len(X_val) == 4500
assert len(X_test) == 4500

print("\nDataset split verification passed!")


Training observations: 21000
Validation observations: 4500
Testing observations: 4500

Target class distribution:
Train {0: 0.7788, 1: 0.2212}
Validation {0: 0.7789, 1: 0.2211}
Test {0: 0.7787, 1: 0.2213}

Training BILL_AMT1 median: 22225.5

Dataset split verification passed!
